# Energy Reconstruction Using CNN - Both Charges and Cos(Zenith)

In [1]:
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt

from datetime import datetime

from tensorflow import keras
from keras import layers
from keras import models
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
#from tensorflow.keras.callbacks import ModelCheckpoint

from data_tools import load_preprocessed, dataPrep, nameModel

simPrefix = os.getcwd()+'\\simdata'

## Data Input

In [2]:
x, y = load_preprocessed(simPrefix, 'train')

Percentage of events with a NaN: 2.68


In [3]:
print(x.shape)
print(y.keys())
# each station has 2 tanks, each tank has 2 DOMs (high/log gain)
# each tank measures charge and time
# each station gives 2 charges and 2 times, 4 total pieces of data per station
# stations arranged in 10x10 square lattice, 2 corners of square unused
# charge measured in VEM, vertical equivalent muon

# 'dir' is true direction, rest of dir are reconstruted by simulations
# 'plane_dir' assumes shower is flat plane
# 'laputop_dir' performs likelihood analysis
# 'small_dir' compromises between plane and laputop

(549773, 10, 10, 4)
dict_keys(['comp', 'energy', 'dir', 'plane_dir', 'laputop_dir', 'small_dir'])


In [4]:
# 85/15 split for training/validation
energy = y['energy']
comp = y['comp']
theta, phi = y['dir'].transpose()
nevents = len(energy)
trainCut = (np.random.uniform(size=nevents) < 0.85)
testCut = np.logical_not(trainCut)

## Model Training

### Alpha Model
- Input: no charge merge, no time layers included, normalized data, combined with zenith angle
- Layers: Two convolutional layers for charge, then combined with zenith
- Output: Energy

In [12]:
# Name for model
key = 'q1q2Z'
numepochs = 6
# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1
prep = {'q':None, 't':False, 'normed':True, 'reco':'', 'cosz':False}

In [6]:
# Create model using functional API for multiple inputs
charge_input=keras.Input(shape=(10,10,2,),name="charge")
conv1_layer = layers.Conv2D(64,kernel_size=3,activation='relu')(charge_input)
conv2_layer = layers.Conv2D(32,kernel_size=3,activation='relu')(conv1_layer)
flat_layer = layers.Flatten()(conv2_layer)

zenith_input=keras.Input(shape=(1,),name="zenith")

concat_layer = layers.concatenate([flat_layer,zenith_input])
output = layers.Dense(1)(concat_layer)

model = models.Model(inputs=[charge_input,zenith_input],outputs=output,name=key)

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

## Old model used for reference
#model = Sequential(name=nameModel(prep, 'test'))  # Automatic naming for flexible assessment later
## Add model layers
#model.add(Conv2D(64, kernel_size=3, activation='relu', input_shape=(10,10,2)))
#model.add(Conv2D(32, kernel_size=3, activation='relu'))
#model.add(Flatten())
#model.add(Dense(1)) # No activation function for last layer of regression model

## Compile model
#model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

In [7]:
# Establish arrays to be trained on
x_i = dataPrep(x, y, **prep)
temp_y = energy

In [8]:
model.summary()

Model: "q1q2cosZ"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
charge (InputLayer)             [(None, 10, 10, 2)]  0                                            
__________________________________________________________________________________________________
conv2d (Conv2D)                 (None, 8, 8, 64)     1216        charge[0][0]                     
__________________________________________________________________________________________________
conv2d_1 (Conv2D)               (None, 6, 6, 32)     18464       conv2d[0][0]                     
__________________________________________________________________________________________________
flatten (Flatten)               (None, 1152)         0           conv2d_1[0][0]                   
___________________________________________________________________________________________

In [9]:
keras.utils.plot_model(model,"model.png")

('You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) ', 'for plot_model/model_to_dot to work.')


In [10]:
# Train
history = model.fit(
    {"charge":x_i[0],"zenith":x_i[1]}, temp_y, epochs=numepochs,validation_split=0.15)

Epoch 1/6
14604/14604 [==============================] - 74s 5ms/step - loss: 0.1516 - mae: 0.2479 - mse: 0.1516 - val_loss: 0.0547 - val_mae: 0.1814 - val_mse: 0.0547
Epoch 2/6
14604/14604 [==============================] - 71s 5ms/step - loss: 0.0604 - mae: 0.1897 - mse: 0.0604 - val_loss: 0.0451 - val_mae: 0.1601 - val_mse: 0.0451
Epoch 3/6
14604/14604 [==============================] - 71s 5ms/step - loss: 0.0522 - mae: 0.1752 - mse: 0.0522 - val_loss: 0.0426 - val_mae: 0.1571 - val_mse: 0.0426
Epoch 4/6
14604/14604 [==============================] - 74s 5ms/step - loss: 0.0475 - mae: 0.1664 - mse: 0.0475 - val_loss: 0.0390 - val_mae: 0.1467 - val_mse: 0.0390
Epoch 5/6
14604/14604 [==============================] - 74s 5ms/step - loss: 0.0453 - mae: 0.1622 - mse: 0.0453 - val_loss: 0.0428 - val_mae: 0.1608 - val_mse: 0.0428
Epoch 6/6
14604/14604 [==============================] - 73s 5ms/step - loss: 0.0435 - mae: 0.1585 - mse: 0.0435 - val_loss: 0.0390 - val_mae: 0.1483 - val_mse:

In [13]:
# Save model to file for easy loading
## WHERE ARE YOU SAVING TO?
model.save('model_%s.h5' % key)
f = open("results.txt", "a")
now = datetime.now()
f.write("{}\t{}\tepochs:{}\tloss:{},{}\n".format(
    now.strftime("%m/%d/%Y %H:%M:%S"),
    key,
    numepochs,
    history.history['loss'][numepochs-1],
    history.history['val_loss'][numepochs-1]
))
f.close()

## Your task

- **Create your own model**
- Replace the model here w/ *simplified* form of Brandon's model (focus: including zenith)
- change the zenith input to cosine(zenith) input